# Семинар 13. Свои классы распределений

На прошлых семинарах мы пользовались готовыми распределениями из `scipy.stats`:
`X.pdf(x)`, `X.cdf(x)`, `X.ppf(q)`, `X.rvs(size=...)`. Сегодня пишем такие классы сами.
Так видно, что стоит за каждым методом: формула, сумма или решение уравнения $F(x)=q$.

1. **Часть 1 — вместе.** Базовый класс `Distribution` с абстрактными методами
   и наследник `Uniform` — равномерное распределение. Сравниваем со SciPy.
2. **Часть 2 — сами.** Пишете свои классы для других законов и проверяете их той же функцией.

**Правило:** внутри своих классов `scipy` не используем, можно `numpy` и модуль `math`.
SciPy нужен только для проверки.

**Запуск.** В Colab загрузите скачанный `.ipynb` через «Файл → Загрузить блокнот».
Выполняйте ячейки по порядку сверху вниз (`Shift+Enter`).

## Подготовка

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Часть 1. Разбираем вместе

### 1.1. Интерфейс: базовый класс

Все распределения в SciPy умеют одно и то же: `rvs`, `pdf`, `pmf`, `cdf`, `sf`, `ppf`, `mean`, `var`.
Такой общий набор методов называют **интерфейсом**. Запишем его как класс `Distribution`.

`ABC` и декоратор `@abstractmethod` из модуля `abc` говорят Python:
«у этого класса методы только объявлены, а написать их обязан каждый наследник».

In [ ]:
from abc import ABC, abstractmethod


class Distribution(ABC):
    """Общий интерфейс распределения — те же методы, что у scipy.stats.

    Договорённости для всех наследников:
    * pdf, pmf, cdf, sf, ppf принимают число или массив NumPy
      и возвращают число или массив той же формы;
    * pmf(x) = P(X = x); у непрерывного закона это 0 в каждой точке;
    * плотности у дискретного закона нет: pdf бросает NotImplementedError;
    * rvs(size, random_state) возвращает массив из size наблюдений,
      одинаковый random_state даёт одинаковую выборку.
    """

    @abstractmethod
    def rvs(self, size=1, random_state=None):
        """Выборка из size независимых наблюдений."""

    @abstractmethod
    def pdf(self, x):
        """Плотность f(x)."""

    @abstractmethod
    def pmf(self, x):
        """Вероятность P(X = x)."""

    @abstractmethod
    def cdf(self, x):
        """Функция распределения F(x) = P(X <= x)."""

    @abstractmethod
    def sf(self, x):
        """Функция выживания P(X > x)."""

    @abstractmethod
    def ppf(self, q):
        """Квантиль уровня q: наименьшее x, при котором F(x) >= q."""

    @abstractmethod
    def mean(self):
        """Математическое ожидание E(X)."""

    @abstractmethod
    def var(self):
        """Дисперсия Var(X)."""

    # Обычные (не абстрактные) методы наследники получают готовыми
    def std(self):
        """Стандартное отклонение — через var(), который напишет наследник."""
        return np.sqrt(self.var())

    def __repr__(self):
        params = ", ".join(f"{name}={value}" for name, value in vars(self).items())
        return f"{type(self).__name__}({params})"

Создать объект абстрактного класса нельзя: непонятно, что считать.
Запустите ячейку и прочитайте последнюю строку ошибки.

In [ ]:
Distribution()

Наследник, который написал не все методы, тоже не создаётся. Python перечислит, чего не хватает.

In [ ]:
class Coin(Distribution):
    def mean(self):
        return 0.5

Coin()

### 1.2. Наследник: равномерное распределение $U[a,b]$

| Метод | Формула |
|---|---|
| `pdf(x)` | $f(x)=\dfrac{1}{b-a}$ при $a\le x\le b$, иначе $0$ |
| `cdf(x)` | $F(x)=\dfrac{x-a}{b-a}$, обрезанное до отрезка $[0,1]$ |
| `sf(x)` | $P(X>x)=1-F(x)$ |
| `ppf(q)` | решаем $F(x)=q$: $x=a+q\,(b-a)$ |
| `mean()`, `var()` | $E(X)=\dfrac{a+b}{2}$, $\mathrm{Var}(X)=\dfrac{(b-a)^2}{12}$ |
| `rvs(size)` | $u$ равномерно на $[0,1)$, затем `ppf(u)` |

`class Uniform(Distribution)` — `Uniform` наследует от `Distribution`.
`__init__` запускается при создании объекта и сохраняет параметры в `self`.
Каждый метод принимает число или массив: сначала `np.asarray(x, dtype=float)`,
дальше только операции NumPy, которые работают сразу со всем массивом.

In [ ]:
class Uniform(Distribution):
    """Непрерывное равномерное распределение на отрезке [a, b]."""

    def __init__(self, a, b):
        if not a < b:
            raise ValueError("нужно a < b")
        self.a = a
        self.b = b

    def pdf(self, x):
        x = np.asarray(x, dtype=float)
        inside = (x >= self.a) & (x <= self.b)    # True внутри отрезка
        return inside / (self.b - self.a)         # True -> 1/(b-a), False -> 0

    def pmf(self, x):
        # непрерывный закон: P(X = x) = 0 в любой точке
        x = np.asarray(x, dtype=float)
        return 0.0 * x                            # нули той же формы, что x

    def cdf(self, x):
        x = np.asarray(x, dtype=float)
        return np.clip((x - self.a) / (self.b - self.a), 0, 1)

    def sf(self, x):
        return 1 - self.cdf(x)

    def ppf(self, q):
        q = np.asarray(q, dtype=float)
        return self.a + q * (self.b - self.a)     # решаем F(x) = q

    def mean(self):
        return (self.a + self.b) / 2

    def var(self):
        return (self.b - self.a) ** 2 / 12

    def rvs(self, size=1, random_state=None):
        rng = np.random.default_rng(random_state)
        u = rng.random(size)                      # равномерные на [0, 1)
        return self.ppf(u)                        # растягиваем на [a, b]

### 1.3. Пользуемся своим классом

Перед запуском каждой ячейки скажите, что она выведет.

In [ ]:
U = Uniform(2, 5)
U

In [ ]:
U.mean(), U.var(), U.std()

In [ ]:
U.pdf(3), U.cdf(3), U.sf(3), U.ppf(0.5)

In [ ]:
xs = np.array([1, 2, 3.5, 5, 6])
U.cdf(xs)

In [ ]:
sample = U.rvs(size=100_000, random_state=2026)
sample[:5], sample.mean(), sample.var()

In [ ]:
grid = np.linspace(1, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(sample, bins=50, density=True, alpha=0.35, label="выборка")
axes[0].plot(grid, U.pdf(grid), lw=2, label="pdf")
axes[0].set_title("Плотность")
axes[1].plot(grid, U.cdf(grid), lw=2)
axes[1].set_title("Функция распределения")
for ax in axes:
    ax.grid(alpha=0.3)
axes[0].legend()
plt.show()

### 1.4. Сравниваем со SciPy

В SciPy равномерный закон задаётся не концами отрезка, а сдвигом и длиной:
`stats.uniform(loc=a, scale=b - a)`. Готовая функция `check(my, ref)` сравнивает
все методы вашего объекта `my` с распределением SciPy `ref` на сетке точек и проверяет выборку.

In [ ]:
# @title Готовая функция check (выполните ячейку, разбирать код не нужно)
def check(my, ref, n=200_000, seed=2026):
    """Сравнивает ваше распределение my с распределением SciPy ref.

    my  — объект вашего класса, например Uniform(2, 5);
    ref — «замороженное» распределение SciPy с теми же параметрами.
    """
    discrete = isinstance(ref.dist, stats.rv_discrete)
    lo, hi = ref.ppf(0.001), ref.ppf(0.999)
    if discrete:
        xs = np.arange(lo - 2, hi + 2.5, 0.5)
    else:
        pad = 0.1 * (hi - lo)
        xs = np.linspace(lo - pad, hi + pad, 101)
    qs = np.linspace(0.01, 0.99, 99)
    rows = []

    def row(name, func):
        try:
            ok, note = func()
        except NotImplementedError:
            ok, note = False, "NotImplementedError (метод не дописан?)"
        except Exception as err:
            ok, note = False, f"{type(err).__name__}: {err}"
        rows.append((name, ok, note))

    def same(mine, theirs):
        mine = np.asarray(mine, dtype=float)
        if mine.shape != np.shape(theirs):
            return False, f"форма {mine.shape}, а нужна {np.shape(theirs)}"
        err = np.max(np.abs(mine - theirs)) if mine.size else 0.0
        return bool(np.allclose(mine, theirs, atol=1e-6)), f"max |разница| = {err:.2g}"

    if discrete:
        row("pmf", lambda: same(my.pmf(xs), ref.pmf(xs)))

        def pdf_absent():
            try:
                my.pdf(xs)
            except NotImplementedError:
                return True, "NotImplementedError, как договорились"
            return False, "у дискретного закона pdf должен бросать NotImplementedError"
        rows.append(("pdf", *pdf_absent()))
    else:
        row("pdf", lambda: same(my.pdf(xs), ref.pdf(xs)))
        row("pmf", lambda: same(my.pmf(xs), np.zeros_like(xs)))
    row("cdf", lambda: same(my.cdf(xs), ref.cdf(xs)))
    row("sf", lambda: same(my.sf(xs), ref.sf(xs)))
    row("ppf", lambda: same(my.ppf(qs), ref.ppf(qs)))
    row("mean", lambda: same(my.mean(), ref.mean()))
    row("var", lambda: same(my.var(), ref.var()))
    def scalar_in():
        ok = np.ndim(my.cdf(xs[len(xs) // 2])) == 0
        return ok, "cdf(число) вернул число" if ok else "cdf(число) должен вернуть число"
    row("число на входе", scalar_in)

    def sample():
        s = np.asarray(my.rvs(size=n, random_state=seed), dtype=float)
        if s.shape != (n,):
            return False, f"rvs(size={n}) вернул форму {s.shape}"
        again = np.asarray(my.rvs(size=n, random_state=seed), dtype=float)
        if not np.array_equal(s, again):
            return False, "одинаковый random_state дал разные выборки"
        se = np.sqrt(ref.var() / n)                    # стандартная ошибка среднего
        ok_mean = abs(s.mean() - ref.mean()) < 5 * se
        ok_var = abs(s.var() / ref.var() - 1) < 0.05
        return bool(ok_mean and ok_var), (f"выборочные E = {s.mean():.4f}, "
                                          f"Var = {s.var():.4f}")
    row("rvs", sample)

    width = max(len(name) for name, _, _ in rows)
    for name, ok, note in rows:
        print(f"{'✓' if ok else '✗'}  {name:<{width}}  {note}")
    passed = all(ok for _, ok, _ in rows)
    print("\nВсё совпадает со SciPy." if passed else "\nЕсть расхождения — смотрите строки с ✗.")

In [ ]:
check(Uniform(2, 5), stats.uniform(loc=2, scale=3))

**Про `pmf` у непрерывного закона.** У `stats.uniform` метода `pmf` нет вовсе:

In [ ]:
stats.uniform(loc=2, scale=3).pmf(3)

В нашем интерфейсе `pmf` обязателен для всех, поэтому мы договорились
(см. документацию `Distribution`):

* `pmf(x)` — это $P(X=x)$. У непрерывного закона эта вероятность равна $0$ в каждой точке,
  поэтому `Uniform.pmf` честно возвращает нули;
* плотности у дискретного закона нет, поэтому `pdf` дискретного закона бросает
  `NotImplementedError` — так Python сообщает «этот метод здесь не имеет смысла».

## Часть 2. Сами

В каждом задании напишите класс-наследник `Distribution` со всеми методами интерфейса
и проверьте его функцией `check`, подобрав распределение SciPy с теми же параметрами.
Все проверки должны дать ✓.

Методы принимают число или массив и возвращают результат той же формы.

### Задание 1. Дискретное равномерное

`DiscreteUniform(a, b)`: целые числа $a, a+1, \dots, b$ (оба конца входят), все равновероятны.
Например, `DiscreteUniform(1, 6)` — игральная кость.

Аналог в SciPy: [scipy.stats.randint](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.randint.html).

In [ ]:
# ваш класс DiscreteUniform

In [ ]:
# проверка

### Задание 2. Показательное

`Exponential(lam)` — показательное распределение с параметром $\lambda>0$ (интенсивность).

Аналог в SciPy: [scipy.stats.expon](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.expon.html).

In [ ]:
# ваш класс Exponential

In [ ]:
# проверка

### Задание 3. Биномиальное

`Binomial(n, p)` — число успехов в $n$ независимых испытаниях с вероятностью успеха $p$.

Аналог в SciPy: [scipy.stats.binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html).

Структура данных: при создании объекта сохраните в нём массив вероятностей
$P(X=k)$ для $k=0,1,\dots,n$; остальные методы пусть берут значения из него.

In [ ]:
# ваш класс Binomial

In [ ]:
# проверка

### Задание 4*. Нормальное

`Normal(mu, sigma)` — нормальное распределение с $E(X)=\mu$ и стандартным отклонением $\sigma$.

Аналог в SciPy: [scipy.stats.norm](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html).

In [ ]:
# ваш класс Normal

In [ ]:
# проверка

## Итог

Ответьте письменно в этой ячейке.

1. Какие методы в ваших классах получились одинаковыми? Что можно было бы перенести в базовый класс?
2. В каких классах `ppf` нашёлся формулой, а в каких пришлось искать ответ иначе?
3. Почему одной строки `self.ppf(rng.random(size))` хватает, чтобы получить выборку из любого закона?

*Ответы:* …